# AI-Assisted Customer Support Analysis and Retrieval from Twitter

## Data Exploration

This notebook explores the Customer Support on Twitter dataset.

The raw dataset file `twcs.csv` is not stored in GitHub because it is too large.  
Instead, it should be downloaded from Kaggle and placed locally in the `data/` folder.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

### Dataset Overview

Before starting preprocessing, we first examine the structure of the dataset.  
This helps us understand what kind of data we are working with and which columns are useful for our task.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [3]:
df = pd.read_csv("../Data/twcs.csv")

df.head()
df.shape
df.columns
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                str    
 2   inbound                  bool   
 3   created_at               str    
 4   text                     str    
 5   response_tweet_id        str    
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), str(4)
memory usage: 131.4 MB


The dataset contains over 2.8 million tweets, which is quite large.  
This means we will need to reduce the dataset size later for faster processing.

## Filtering Customer Tweets

The dataset contains both customer messages and company replies.  
Since the goal is to analyze customer support requests, we only keep tweets written by customers.

This is done using the `inbound` column:
- True → customer tweet  
- False → company response  

In [7]:
customer_df = df[df["inbound"] == True].copy()

customer_df.shape


(1537843, 7)

In [8]:

df["inbound"].value_counts()

inbound
True     1537843
False    1273931
Name: count, dtype: int64

## Data Cleaning

The dataset is still large after filtering customer tweets.  
To make processing more efficient, we perform basic cleaning and reduce the dataset size.

Steps:
- remove missing text
- remove duplicate tweets
- sample a smaller subset for faster experimentation

In [11]:
# Removing missing text
customer_df = customer_df.dropna(subset=["text"])

# Removing duplicates
customer_df["text"].duplicated().sum()
customer_df = customer_df.drop_duplicates(subset=["text"])

customer_df.shape

(1511776, 7)

In [12]:
#Reduce Dataset
customer_df = customer_df.sample(n=200000, random_state=42)

customer_df.shape

(200000, 7)

## Extracting Company Labels

The dataset does not contain a direct category label.  
To create a supervised classification task, we use company mentions in the tweet text.

In [ ]:
#Extract mentions
def extract_mentions(text):
    return re.findall(r"@(\w+)", str(text).lower())

customer_df["mentions"] = customer_df["text"].apply(extract_mentions)

customer_df[["text", "mentions"]].head(10)

,text,mentions
177501,@AmericanAir Thanks. I still don't know why it...,[americanair]
1105226,I've been silver status on @Delta for 6 flight...,[delta]
559784,@ChaseSupport @116016 @ChaseSupport when shoul...,"[chasesupport, 116016, chasesupport]"
2218147,@AskPlayStation Ooohhh ok because other games ...,[askplaystation]
1723302,@AskPayPal I don't know what to do. Could you ...,[askpaypal]
326678,@AppleSupport A month or 2,[applesupport]
1878609,wow mcdonald' s hot &amp; spicy truly holds th...,[]
2620195,@115940 What song is being sung on Ep. 2 of #M...,[115940]
2729013,"@ArgosHelpers Also, i hope i have no trouble b...",[argoshelpers]
2791668,Applying to @116297 and their assessment has w...,[116297]


In [14]:
#Keep tweets with mentions
customer_df = customer_df[customer_df["mentions"].apply(len) > 0].copy()

customer_df.shape

(192533, 8)

In [15]:
#Create label
customer_df["company"] = customer_df["mentions"].apply(lambda x: x[0])

customer_df[["text", "company"]].head(10)

,text,company
177501,@AmericanAir Thanks. I still don't know why it...,americanair
1105226,I've been silver status on @Delta for 6 flight...,delta
559784,@ChaseSupport @116016 @ChaseSupport when shoul...,chasesupport
2218147,@AskPlayStation Ooohhh ok because other games ...,askplaystation
1723302,@AskPayPal I don't know what to do. Could you ...,askpaypal
326678,@AppleSupport A month or 2,applesupport
2620195,@115940 What song is being sung on Ep. 2 of #M...,115940
2729013,"@ArgosHelpers Also, i hope i have no trouble b...",argoshelpers
2791668,Applying to @116297 and their assessment has w...,116297
484678,"@247619 Hi Glulia, this is disappointing to he...",247619


In [20]:
#Top companies
customer_df["company"].value_counts().head(15)

#customer_df.shape

company
amazonhelp         16230
applesupport       10964
americanair         6049
delta               5385
uber_support        5344
southwestair        4311
115858              4285
virgintrains        4257
tesco               3888
spotifycares        3823
british_airways     3791
gwrhelp             3194
xboxsupport         3111
askplaystation      2752
chipotletweets      2704
Name: count, dtype: int64

## Selecting Top Companies

Some companies appear very rarely in the dataset.  
To make the classification more balanced and efficient, we will keep only top 10 most frequent companies.

In [ ]:
# Top 10 most frequent companies
top_companies = customer_df["company"].value_counts().head(10).index
top_companies

# Filter dataset to keep only these companies
customer_df = customer_df[customer_df["company"].isin(top_companies)].copy()

# Check dataset size
customer_df.shape


(64536, 9)


In [ ]:
# Show how many samples each company has
customer_df["company"].value_counts()

company
amazonhelp      16230
applesupport    10964
americanair      6049
delta            5385
uber_support     5344
southwestair     4311
115858           4285
virgintrains     4257
tesco            3888
spotifycares     3823
Name: count, dtype: int64